# 00. セットアップ — 環境構築・認証・パス設定

1. **Google Drive マウント**
2. **GitHub リポジトリ取得 / 最新化** (GITHUB_TOKEN必須)
3. **pip install**
4. **環境変数チェック** (ANTHROPIC_API_KEY)
5. **パス設定**

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
print(f"IN_COLAB={IN_COLAB}")

In [ ]:
# GitHub clone / pull (GITHUB_TOKEN をシークレットに登録)
REPO_NAME  = "AI_TradeManagement"
REPO_OWNER = "tsp0918"

if IN_COLAB:
    import subprocess
    _token = ""
    try:
        from google.colab import userdata
        _token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        _token = os.environ.get("GITHUB_TOKEN", "")
    if not _token:
        raise RuntimeError("GITHUB_TOKEN が未設定です")
    REPO_URL = f"https://{_token}@github.com/{REPO_OWNER}/{REPO_NAME}"
    r = subprocess.run(
        f"git -C /content/{REPO_NAME} pull || git clone {REPO_URL} /content/{REPO_NAME}",
        shell=True, capture_output=True, text=True)
    print((r.stdout or r.stderr).replace(_token, "***"))
    REPO_BASE = f"/content/{REPO_NAME}"
else:
    import pathlib
    REPO_BASE = "/Users/takehirosato/Desktop/AI_TradeManagement"

print(f"REPO_BASE={REPO_BASE}")
sys.path.insert(0, f"{REPO_BASE}/scripts")

In [ ]:
\!pip install -q requests anthropic sentence-transformers faiss-cpu

In [ ]:
# APIキー設定 (BIS_API_KEY は別々の try/except で取得)
import os
ANTHROPIC_API_KEY = ""
BIS_API_KEY = "DEMO_KEY"

try:
    from google.colab import userdata
    _val = userdata.get("ANTHROPIC_API_KEY")
    if _val:
        ANTHROPIC_API_KEY = _val
        print("[source] Colab Secrets")
    else:
        print("[warn] userdata.get() returned None — トグルがONか確認して再実行")
except Exception as _e:
    print(f"[warn] {_e}")

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

try:
    from google.colab import userdata as _ud2
    BIS_API_KEY = _ud2.get("BIS_API_KEY") or "DEMO_KEY"
except Exception:
    pass

if ANTHROPIC_API_KEY:
    print(f"✅ ANTHROPIC_API_KEY: {chr(42)*(len(ANTHROPIC_API_KEY)-4)}{ANTHROPIC_API_KEY[-4:]}")
else:
    print("⚠️ ANTHROPIC_API_KEY が取得できません。シークレットのトグルをONにして再実行してください")
print(f"✅ BIS_API_KEY: {BIS_API_KEY}")

In [ ]:
from pathlib import Path
BASE = Path(REPO_BASE)
STAGING_DIR = BASE / "data" / "staging"
UNIFIED_DIR = BASE / "data" / "unified"
for sub in ("sanctions","fefta","eccn","patents","mappings","faiss"):
    (STAGING_DIR / sub).mkdir(parents=True, exist_ok=True)
print(f"BASE: {BASE}  ({chr(79)+chr(75) if BASE.exists() else chr(78)+chr(79)+chr(84)+chr(32)+chr(70)+chr(79)+chr(85)+chr(78)+chr(68)})")
print(f"STAGING_DIR: {STAGING_DIR}  ({chr(79)+chr(75) if STAGING_DIR.exists() else chr(78)+chr(79)+chr(84)+chr(32)+chr(70)+chr(79)+chr(85)+chr(78)+chr(68)})")
print("✅ セットアップ完了")